# Veritas AI for PBL: Long-Context Fake News Classifier

**Filename:** `veritas_ai_pbl_notebook.ipynb`
**Author / Team:** Veritas AI (Pushkar Kumar & team)
**Purpose:** Train a long-context transformer model (`Longformer`) to classify news articles as REAL or FAKE. This version saves all artifacts locally to Google Drive and launches a Gradio demo directly from Colab.

**Workflow:**
* Mount Google Drive and set up paths.
* Load and preprocess the ISOT dataset from Drive.
* Fine-tune `Longformer` to handle up to 1024 tokens.
* Save the final model and tokenizer to a new `SEM_3_PBL` folder in Drive.
* Launch an interactive Gradio web UI from within this notebook.

**Prerequisite:**
* The `Fake.csv` and `True.csv` files must be uploaded to `/content/drive/MyDrive/SEM_3_PBL/`.

In [ ]:
# Part 1: 🛠️ Environment & Project Setup
#
# Mount Google Drive to access our files.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Now, we'll define the path to your new `SEM_3_PBL` project directory and verify that the dataset files exist.

In [ ]:
import os

# Define the base directory for the PBL project in your Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/SEM_3_PBL/"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True) # Create the directory if it doesn't exist

# Define paths for the dataset files
FAKE_CSV_PATH = os.path.join(DRIVE_PROJECT_DIR, "Fake.csv")
TRUE_CSV_PATH = os.path.join(DRIVE_PROJECT_DIR, "True.csv")

# Verify that both files exist
if not os.path.exists(FAKE_CSV_PATH) or not os.path.exists(TRUE_CSV_PATH):
    raise FileNotFoundError("One or both dataset files (Fake.csv, True.csv) were not found in the specified Drive folder.")
else:
    print("Successfully located Fake.csv and True.csv in Google Drive.")

Successfully located Fake.csv and True.csv in Google Drive.


Next, we install the necessary libraries. This includes `transformers`, `datasets`, and `gradio` for our final UI. We do not need to install `huggingface_hub` for this offline workflow.

**Note:** After this cell runs, you may need to restart the runtime (`Runtime > Restart runtime`) for the changes to take full effect.

In [ ]:
# Install compatible libraries
!pip install transformers==4.38.1 datasets==2.18.0 accelerate==0.28.0 peft==0.9.0 gradio==3.47.1 websockets==11.0.3 -q

With the setup complete, we load the `True.csv` file from its verified Google Drive path into a Pandas DataFrame.

In [ ]:
# Part 2: 💾 Data Loading & Preparation
#
import pandas as pd

df_true = pd.read_csv(TRUE_CSV_PATH)
df_fake = pd.read_csv(FAKE_CSV_PATH)

We'll now add our `label` column (`0` for REAL, `1` for FAKE), combine the two DataFrames, and shuffle the result to ensure our data is not ordered.

In [ ]:
df_true['label'] = 0
df_fake['label'] = 1
df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- Combined and Shuffled DataFrame Sample ---")
df.head()

--- Combined and Shuffled DataFrame Sample ---


,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1


For our NLP model, we combine the `title` and `text` into a single `full_text` column and drop the columns we won't use for training to prevent data leakage.

In [ ]:
df['full_text'] = df['title'] + " " + df['text']
df_model = df[['full_text', 'label']].copy()

print("--- Final DataFrame for Modeling ---")
df_model.head()

--- Final DataFrame for Modeling ---


,full_text,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,1
1,Failed GOP Candidates Remembered In Hilarious...,1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,1
3,California AG pledges to defend birth control ...,0
4,AZ RANCHERS Living On US-Mexico Border Destroy...,1


In [ ]:
# NEW CELL: Subsample to about 1024 examples total
df_small = df_model.sample(n=10024, random_state=42)

print("Original size:", len(df_model))
print("Subsampled size:", len(df_small))
print(df_small["label"].value_counts())


Original size: 44898
Subsampled size: 10024
label
1    5241
0    4783
Name: count, dtype: int64


Now, we split our prepared data into training and testing sets with an 80/20 ratio.

In [ ]:
# Part 3: ✂️ Data Splitting & Conversion (using smaller df_small)
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_small,                # was df_model
    test_size=0.2,
    random_state=42,
    stratify=df_small["label"],
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Train label counts:")
print(train_df["label"].value_counts())
print("\nTest label counts:")
print(test_df["label"].value_counts())


Train size: 8019
Test size: 2005
Train label counts:
label
1    4193
0    3826
Name: count, dtype: int64

Test label counts:
label
1    1048
0     957
Name: count, dtype: int64


Next, we convert our Pandas DataFrames into the Hugging Face `Dataset` format, which is required by the `Trainer` API.

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

raw_datasets = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['full_text', 'label', '__index_level_0__'],
        num_rows: 8019
    })
    test: Dataset({
        features: ['full_text', 'label', '__index_level_0__'],
        num_rows: 2005
    })
})


**Model Change:** To handle long articles, we are switching from `DistilBERT` (512 token limit) to `Longformer`. This model can process up to 4096 tokens, allowing us to analyze much more of each news article.

In [ ]:
# Part 4: ✍️ Tokenization for Longformer
#
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "allenai/longformer-base-4096"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


This function defines our tokenization logic, now with a `max_length` of 4096 to match the Longformer model's capability.

In [ ]:
def tokenize_function(examples):
    # Longformer uses a global attention mask for the [CLS] token, let's ensure it's set
    inputs = tokenizer(examples["full_text"], padding="max_length", truncation=True, max_length=1024)

    # Create global attention mask
    global_attention_mask = [0] * len(inputs['input_ids'])
    global_attention_mask[0] = 1 # Set global attention on the first token ([CLS])
    inputs['global_attention_mask'] = global_attention_mask

    return inputs

We apply the tokenization function across our entire dataset. This may take longer than before due to the increased text length.

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/8019 [00:00<?, ? examples/s]

Map:   0%|          | 0/2005 [00:00<?, ? examples/s]

Now we load the `Longformer` model itself, configured for a binary classification task.

In [ ]:
# Part 5: 🧠 Model Training with Longformer
#
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
)

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


We configure our `TrainingArguments`. **Important Changes:**
* `push_to_hub` is set to `False`.
* `output_dir` points to a local directory.
* `per_device_train_batch_size` is reduced to `1` because Longformer is very memory-intensive.
* `gradient_accumulation_steps` is added to compensate for the small batch size. This effectively simulates a larger batch size.

In [ ]:
from transformers import TrainingArguments

# Change OUTPUT_DIR to a subfolder within DRIVE_PROJECT_DIR to save to Drive
OUTPUT_DIR = "/content/drive/MyDrive/SEM_3_PBL/veritas_ai_pbl_longformer_checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # a bit smaller so steps are faster
    gradient_accumulation_steps=4,

    num_train_epochs=1,
    weight_decay=0.01,

    # --- MEMORY / SPEED OPTIMIZATION ---
    fp16=True,                   # enable mixed precision
    gradient_checkpointing=True, # reduce memory, allow larger models

    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none",
)

The `Trainer` object brings everything together for our local training run.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


This command starts the fine-tuning process. Be aware, training Longformer will be significantly slower than DistilBERT.

In [ ]:
print("Starting Longformer model training... This will take some time.")
trainer.train()

Starting Longformer model training... This will take some time.


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,0.005100,0.000013


TrainOutput(global_step=2004, training_loss=0.017658273408750813, metrics={'train_runtime': 2065.1215, 'train_samples_per_second': 3.883, 'train_steps_per_second': 0.97, 'total_flos': 5265322518970368.0, 'train_loss': 0.017658273408750813, 'epoch': 1.0})

In [ ]:
# Part 6: 💾 Save Model and Tokenizer to Google Drive
#
import os

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the fine-tuned model and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model and tokenizer saved to: {OUTPUT_DIR}")

# Verify files are saved
print("Contents of saved model directory:")
for item in os.listdir(OUTPUT_DIR):
    print(f"- {item}")

Model and tokenizer saved to: /content/drive/MyDrive/SEM_3_PBL/veritas_ai_pbl_longformer_checkpoints
Contents of saved model directory:
- config.json
- model.safetensors
- tokenizer_config.json
- special_tokens_map.json
- vocab.json
- merges.txt
- tokenizer.json
- training_args.bin


To download the model:

1.  **Open Google Drive**: Go to `drive.google.com` in your web browser.
2.  **Navigate to the directory**: Find the folder `SEM_3_PBL` in your 'My Drive' section, then open `veritas_ai_pbl_longformer_checkpoints`.
3.  **Download**: You can download individual files or the entire folder by right-clicking and selecting 'Download'.

In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/SEM_3_PBL/veritas_ai_pbl_longformer_checkpoints"

print(f"Contents of the model directory in Google Drive ({OUTPUT_DIR}):")
if os.path.exists(OUTPUT_DIR):
    for item in os.listdir(OUTPUT_DIR):
        print(f"- {item}")
else:
    print("Model directory not found. Please ensure it was saved correctly.")

Contents of the model directory in Google Drive (/content/drive/MyDrive/SEM_3_PBL/veritas_ai_pbl_longformer_checkpoints):
- config.json
- model.safetensors
- tokenizer_config.json
- special_tokens_map.json
- vocab.json
- merges.txt
- tokenizer.json
- training_args.bin


## Part 7: 🚀 Launch Gradio Web UI

Now, we'll set up and launch a Gradio interface to interact with our fine-tuned Longformer model.

In [ ]:
# Load the saved model and tokenizer for inference
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# MODEL_CHECKPOINT is already defined, but we'll use OUTPUT_DIR to load the fine-tuned model
loaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval() # Set model to evaluation mode

print(f"Model and tokenizer loaded from: {OUTPUT_DIR}")

Model and tokenizer loaded from: /content/drive/MyDrive/SEM_3_PBL/veritas_ai_pbl_longformer_checkpoints


In [ ]:
def predict_fake_news(text):
    inputs = loaded_tokenizer(text, padding="max_length", truncation=True, max_length=1024, return_tensors="pt")

    # Add global attention mask for Longformer
    global_attention_mask = torch.zeros(inputs['input_ids'].shape, dtype=torch.long, device=device)
    global_attention_mask[:, 0] = 1 # Set global attention on the first token ([CLS])
    inputs['global_attention_mask'] = global_attention_mask

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

    # Class labels: 0 for REAL, 1 for FAKE
    if probabilities[1] > probabilities[0]:
        return f"FAKE News (Confidence: {probabilities[1]:.2f})"
    else:
        return f"REAL News (Confidence: {probabilities[0]:.2f})"

In [ ]:
import gradio as gr

# Create the Gradio interface
iface = gr.Interface(
    fn=predict_fake_news,
    inputs=gr.Textbox(lines=5, placeholder="Enter news article text here..."),
    outputs="text",
    title="Long-Context Fake News Classifier (Longformer)",
    description="Enter a news article to classify it as REAL or FAKE.",
    examples=[
        ["BREAKING: Donald Trump has just signed a new executive order to ban all social media from the United States, citing national security concerns."],
        ["President Biden today announced new initiatives aimed at boosting renewable energy production and creating green jobs across the country."]
    ]
)

print("Launching Gradio interface...")
iface.launch(share=True) # share=True generates a public link

IMPORTANT: You are using gradio version 3.47.1, however version 4.44.1 is available, please upgrade.
--------
Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://1b489670a95372bc6a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def greet(name):
    return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()
